# Lab 1 - Data Setup

**Objective:** load the Kaggle Airline dataset into a managed Iceberg table on Cloudera Data Warehouse.

**Estimated time:** ~15 minutes.

**You will:**

1. Connect to your Trino Virtual Warehouse using workload credentials.
2. Create the `airline_lab` schema in both the `hive` and `iceberg` catalogs.
3. Register an *external* staging table over the raw CSV.
4. Populate the *managed* Iceberg target table `iceberg.airline_lab.flights`.

**Prerequisites:** complete *Lab 0 - Before you begin* in the README and confirm the `TRINO_*` environment variables are exported in the shell that launched JupyterLab.

**Verification:** at the end, `SELECT COUNT(*) FROM iceberg.airline_lab.flights` returns a non-zero count.

In [1]:
import os
from trino.dbapi import connect
from trino.auth import BasicAuthentication

TRINO_HOST     = os.environ['TRINO_HOST']
TRINO_PORT     = int(os.environ.get('TRINO_PORT', 443))
TRINO_USER     = os.environ['TRINO_USER']
TRINO_PASSWORD = os.environ['TRINO_PASSWORD']
TRINO_CATALOG  = os.environ.get('TRINO_CATALOG', 'iceberg')
TRINO_SCHEMA   = os.environ.get('TRINO_SCHEMA', 'airline_lab')

conn = connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user=TRINO_USER,
    auth=BasicAuthentication(TRINO_USER, TRINO_PASSWORD),
    http_scheme='https',
    catalog=TRINO_CATALOG,
    schema=TRINO_SCHEMA,
)
cur = conn.cursor()
cur.execute('SELECT current_user, current_catalog, current_schema')
cur.fetchone()

['aktiwari', 'iceberg', 'airline_lab']

## Step 1 - Create schemas

The `hive` catalog holds the external staging table that points at the raw CSV on object storage. The `iceberg` catalog holds the curated, partitioned target table. 

Update the value of `RAW_LOCATION` by replacing the `<bucket_name>` to match a bucket or container your CDW environment has write access to.

In [2]:
RAW_LOCATION = 's3a://psehol-env-buk-b1991c3b/airline_lab/'   # <-- edit

cur.execute(f"CREATE SCHEMA IF NOT EXISTS hive.airline_lab WITH (location = '{RAW_LOCATION}')")
cur.fetchall()

cur.execute('CREATE SCHEMA IF NOT EXISTS iceberg.airline_lab')
cur.fetchall()
print('Schemas ready.')

Schemas ready.


## Step 2 - Register the external staging table

The staging table is backed by the CSV you uploaded in `Lab 1 -> Step 1` of the README. It is external, so dropping it does not delete the source file. 

Update the value of `external_location` by replacing the `<bucket_name>` to match a bucket or container your CDW environment has write access to.

In [3]:
cur.execute("""
CREATE TABLE IF NOT EXISTS hive.airline_lab.flights_raw (
    passenger_id          VARCHAR,
    first_name            VARCHAR,
    last_name             VARCHAR,
    gender                VARCHAR,
    age                   VARCHAR,
    nationality           VARCHAR,
    airport_name          VARCHAR,
    airport_country_code  VARCHAR,
    country_name          VARCHAR,
    airport_continent     VARCHAR,
    continents            VARCHAR,
    departure_date        VARCHAR,
    arrival_airport       VARCHAR,
    pilot_name            VARCHAR,
    flight_status         VARCHAR
)
WITH (
    format = 'CSV',
    external_location = 's3a://psehol-env-buk-b1991c3b/airline_lab/raw/',
    skip_header_line_count = 1
)
""")
cur.fetchall()

[]

## Verify
The table has been created by running below. 

In [11]:
cur.execute("SHOW TABLES FROM hive.airline_lab")
print(cur.fetchall())

[['flights'], ['flights_raw']]


## Step 3 - Create the managed Iceberg target table

The table is partitioned by `year(departure_date)` and `airport_continent`. Iceberg applies these partition transforms automatically at write time, and Trino prunes partitions at read time based on the query predicates you write on `departure_date` and `airport_continent`.

In [12]:
cur.execute("""
CREATE TABLE IF NOT EXISTS iceberg.airline_lab.flights (
    passenger_id          VARCHAR,
    first_name            VARCHAR,
    last_name             VARCHAR,
    gender                VARCHAR,
    age                   INTEGER,
    nationality           VARCHAR,
    airport_name          VARCHAR,
    airport_country_code  VARCHAR,
    country_name          VARCHAR,
    airport_continent     VARCHAR,
    continents            VARCHAR,
    departure_date        DATE,
    arrival_airport       VARCHAR,
    pilot_name            VARCHAR,
    flight_status         VARCHAR
)
WITH (
    format = 'PARQUET',
    partitioning = ARRAY['year(departure_date)', 'airport_continent']
)
""")
cur.fetchall()

[]

## Step 4 - Load the data

The `INSERT INTO ... SELECT` parses the CSV date string into a `DATE`, casts types as needed, and writes partitioned Parquet files into the Iceberg table.

In [15]:
cur.execute("""
INSERT INTO iceberg.airline_lab.flights
SELECT
    passenger_id, first_name, last_name, gender,
    CAST(age AS INTEGER) AS age,
    nationality,
    airport_name, airport_country_code, country_name,
    airport_continent, continents,
    CAST(departure_date AS DATE) AS departure_date,
    arrival_airport, pilot_name, flight_status
FROM hive.airline_lab.flights_raw
""")
cur.fetchall()
cur.execute('SELECT COUNT(*) FROM iceberg.airline_lab.flights')
print('Rows loaded:', cur.fetchone()[0])

Rows loaded: 100


## Verify

Expect a non-zero row count. If the count is zero, re-check the `external_location` in Step 2 and the path you uploaded the CSV to.

**Next:** open `02_exploration.ipynb`.